# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 tabular dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}\nVersion: {metadata.version}\nIdentifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

Here, we inspect the record sets defined in the dataset. Each record set and its fields are referenced by their Croissant `@id`.

In [ ]:
# List available record sets and their IDs
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in the metadata.")
else:
    print("Record sets found:")
    for rs in record_sets:
        print(f"- @id: {rs['@id']} | name: {rs.get('name', 'N/A')}")

# For demonstration, let us iterate fields for the first record set
if record_sets:
    rs_id = record_sets[0]['@id']
    # Get fields by @id
    fields = record_sets[0].get('field', [])
    print(f"Fields for record set {rs_id}:")
    for field in fields:
        field_id = field['@id'] if isinstance(field, dict) else field
        print(f"  Field: {field_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the `@id` for each record set. As an example, we use the first record set from the overview above.

In [ ]:
# Extract data from all available record sets using their @id
dataframes = {}
record_set_ids = []

for rs in dataset.metadata.recordSet:
    rs_id = rs['@id']
    record_set_ids.append(rs_id)
    # Load the records from this record set
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print all available record set ids
print("All record set @ids:")
for rid in record_set_ids:
    print(rid)

# Example: Print columns for the first record set
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"Columns in record set {example_rs_id}:")
    print(dataframes[example_rs_id].columns.tolist())
    dataframes[example_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Here, we demonstrate filtering and normalization using the `@id` of numeric fields in the chosen record set.

In [ ]:
# For demonstration, use a numeric field from the first record set
# Here, we search for a numeric field (e.g., 'Age'), which is listed in personalSensitiveInformation
example_rs_id = record_set_ids[0] if record_set_ids else None

if example_rs_id:
    df = dataframes[example_rs_id]
    # Find available numeric columns and pick one
    numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64'] or 'age' in col.lower()]
    if not numeric_fields:
        print("No numeric fields found for EDA.")
    else:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")

        threshold = 50
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by an attribute such as sex or anatomical location
        group_field_candidates = [col for col in df.columns if any(word in col.lower() for word in ['sex', 'anatomical', 'location'])]
        group_field = group_field_candidates[0] if group_field_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we plot the distribution of the chosen numeric field and visualize any relationships across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_rs_id and numeric_fields:
    df = dataframes[example_rs_id]
    # Plot distribution
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we used the Croissant schema and the `mlcroissant` library to load, inspect, and process the FAIR^2 clinical colorectal cancer dataset. By referencing all entities by their `@id`, our workflow ensures reproducibility and clarity. We demonstrated basic EDA and visualizations, preparing the dataset for further statistical or machine learning analysis.